In [1]:
import cv2 as cv
import torch
from ultralytics import YOLO
from collections import deque
import time

In [2]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA GeForce RTX 3080 Ti


In [3]:
model=YOLO("yolo11m.pt")
cap=cv.VideoCapture("../data/3min.mp4")


In [4]:
if not cap.isOpened():
    print("Не удалось открыть видео — проверьте путь или кодек")

In [5]:
colors = {
    "car": (0, 255, 0),
    "bus": (255, 0, 0),
}

In [6]:
#quick remove last item (pop(0))
fps_history=deque(maxlen=30)

In [7]:
_,frame=cap.read()
print(frame.shape)
cv.rectangle(frame,(250,600),(720,850),(255,255,0),3)
cv.imshow("1",frame)
cv.waitKey(0)

: 

In [ ]:
frame[0]

In [ ]:
x1_roi,y1_roi,x2_roi,y2_roi=250,600,720,850

In [ ]:
cv.imshow("1",frame[y1_roi:y2_roi,x1_roi:x2_roi])
cv.waitKey(0)

In [7]:
cap=cv.VideoCapture("../data/3min.mp4")
prev_time=time.time()
while cap.isOpened():
    ret, frame= cap.read()
    results = model.predict(frame[y1_roi:y2_roi,x1_roi:x2_roi],classes=[2,3,5,7], verbose=False)[0]
    # results = model.track(frame[y1_roi:y2_roi,x1_roi:x2_roi],persist=True, classes=[2,3,5,7], verbose=False)[0]
    cur_time=time.time()
    fps_history.append(1/(cur_time-prev_time))
    prev_time=time.time()
    avg_fps=sum(fps_history)/len(fps_history)
    cv.putText(frame, f"FPS: {avg_fps:.1f}",(10,30), cv.FONT_HERSHEY_SIMPLEX,1,(0,255,0),2)

    for result in results.boxes:
        x1,y1, x2,y2=map(int,result.xyxy[0])
        #transform coord from roi_frame to origin frame
        x1, x2 = x1 + x1_roi, x2 + x1_roi
        y1, y2 = y1 + y1_roi, y2 + y1_roi
        
        cv.rectangle(frame,(x1,y1),(x2,y2),(255,0,0),3)
        class_id=int(result.cls[0])
        class_name=model.names[class_id]
        conf=result.conf[0]
        color_ob=colors.get(class_name,(255,255,255))
        track_id = int(result.id[0]) if result.id is not None else -1
        ob_label=f"{class_name} #{track_id}  {conf:.2f}"
        

        cv.putText(frame, ob_label,(x1,y1-8), cv.FONT_HERSHEY_SIMPLEX, 0.5, color_ob, 2)
        cv.imshow("frame",frame)
    if cv.waitKey(1)==ord("q"):
        print("avg_fps")
        break
cap.release()
cv.destroyAllWindows()


NameError: name 'y1_roi' is not defined

In [ ]:
print(results.speed)

In [ ]:
import torch
print(torch.get_num_threads())

In [ ]:
results = model.predict(source="../data/3min.mp4", save=True, stream=True)

In [ ]:
for result in results:
    xywh = result.boxes.xywh  # center-x, center-y, width, height
    xywhn = result.boxes.xywhn  # normalized
    xyxy = result.boxes.xyxy  # top-left-x, top-left-y, bottom-right-x, bottom-right-y
    xyxyn = result.boxes.xyxyn  # normalized
    names = [result.names[cls.item()] for cls in result.boxes.cls.int()]  # class name of each box
    confs = result.boxes.conf  # confidence score of each box

In [ ]:
model.names